# Experiment 3: Iris Flower Classification using K-Nearest Neighbors

**Course:** MTech AI - DTU

**Task Type:** Classification

**Model:** K-Nearest Neighbors (KNN)

**Problem Statement:** Classify flowers into Setosa, Versicolor, and Virginica using sepal and petal measurements.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

## 1. Dataset

The Iris dataset is loaded from `sklearn.datasets`. It contains 150 samples, four numeric features, and three classes with 50 samples in each class.

In [ ]:
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df["species"] = pd.Categorical.from_codes(iris.target, iris.target_names)

df.head()

In [ ]:
df["species"].value_counts()

## 2. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for species in iris.target_names:
    part = df[df["species"] == species]
    axes[0].scatter(part["petal length (cm)"], part["petal width (cm)"], label=species, alpha=0.7)
axes[0].set_xlabel("Petal length (cm)")
axes[0].set_ylabel("Petal width (cm)")
axes[0].set_title("Petal Measurements by Species")
axes[0].legend()

axes[1].boxplot(
    [df.loc[df["species"] == species, "petal length (cm)"] for species in iris.target_names],
    labels=iris.target_names
)
axes[1].set_xlabel("Species")
axes[1].set_ylabel("Petal length (cm)")
axes[1].set_title("Petal Length by Species")

plt.tight_layout()
plt.show()

**Observation:** Setosa is well separated from the other two species in the petal measurements. Versicolor and Virginica have more overlap.

## 3. Preprocessing

KNN is a distance-based algorithm, so the features are standardized. The scaler is placed inside a pipeline so that, during cross-validation, each fold uses only its own training data to calculate the scaling parameters.

In [ ]:
X = df[iris.feature_names]
y = df["species"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

## 4. Choosing K

The test set is kept separate while selecting `k`. Five-fold stratified cross-validation is performed only on the training data.

In [ ]:
k_values = range(1, 21)
cv_scores = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for k in k_values:
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k))
    ])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy")
    cv_scores.append(scores.mean())

best_k = list(k_values)[int(np.argmax(cv_scores))]

plt.figure(figsize=(7, 4.5))
plt.plot(list(k_values), cv_scores, marker="o")
plt.xlabel("k (number of neighbors)")
plt.ylabel("Mean 5-fold CV Accuracy")
plt.title("Cross-Validation Accuracy vs. k")
plt.xticks(list(k_values))
plt.show()

print(f"Best k: {best_k}")
print(f"Mean CV accuracy: {max(cv_scores):.3f}")

## 5. Model Training - KNN

In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=best_k))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {test_accuracy:.3f}")
print()
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=iris.target_names)
ConfusionMatrixDisplay(cm, display_labels=iris.target_names).plot(colorbar=False)
plt.title(f"Confusion Matrix (k={best_k})")
plt.show()

## 6. Conclusion

KNN was used to classify Iris flowers into Setosa, Versicolor, and Virginica. The features were standardized because KNN uses distances between samples. The value of `k` was selected using five-fold cross-validation on the training data, and the final model was evaluated once on the separate test set. Setosa is well separated from the other classes, while most of the classification difficulty is between Versicolor and Virginica.